# Parser Inference FastAPI + ngrok on 

In [ ]:
!pip install -q fastapi "uvicorn[standard]" pyngrok transformers peft accelerate bitsandbytes safetensors pydantic

In [ ]:
import os
import json
import time
import zipfile
import tempfile
from pathlib import Path
from typing import Any
import threading

import torch
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel, Field
import uvicorn
from pyngrok import ngrok
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import PeftModel

In [ ]:
BASE_MODEL_ID = os.getenv("PARSER_BASE_MODEL_ID", "Qwen/Qwen3-4B-Instruct-2507")

ARTIFACT_ZIP_PATH = os.getenv(
    "PARSER_ARTIFACT_ZIP",
    "/kaggle/input/government-parser-qwen3-4b-lora-artifact/government_parser_qwen3_4b_lora_artifact.zip",
)

ADAPTER_PATH = os.getenv(
    "PARSER_ADAPTER_PATH",
    "/kaggle/input/models/vnphongv/government-parser-qwen3-4b-lora-artifact/other/default/1/final_adapter"
)

EXTRACT_DIR = "/kaggle/working/government_parser_artifact"
PORT = int(os.getenv("PARSER_SERVICE_PORT", "8000"))

from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()
NGROK_AUTHTOKEN = user_secrets.get_secret("NGROK_AUTHTOKEN")

MAX_NEW_TOKENS = int(os.getenv("PARSER_MAX_NEW_TOKENS", "512"))
TEMPERATURE = float(os.getenv("PARSER_TEMPERATURE", "0.0"))
TOP_P = float(os.getenv("PARSER_TOP_P", "0.95"))
LOAD_IN_4BIT = os.getenv("PARSER_LOAD_IN_4BIT", "true").lower() == "true"

SERVICE_VERSION = "parser_model_service_v0.1.0"

config_summary = {
    "BASE_MODEL_ID": BASE_MODEL_ID,
    "ARTIFACT_ZIP_PATH": ARTIFACT_ZIP_PATH,
    "ADAPTER_PATH": ADAPTER_PATH,
    "EXTRACT_DIR": EXTRACT_DIR,
    "PORT": PORT,
    "NGROK_AUTHTOKEN_SET": bool(NGROK_AUTHTOKEN),
    "MAX_NEW_TOKENS": MAX_NEW_TOKENS,
    "TEMPERATURE": TEMPERATURE,
    "TOP_P": TOP_P,
    "LOAD_IN_4BIT": LOAD_IN_4BIT,
    "SERVICE_VERSION": SERVICE_VERSION,
}

print(json.dumps(config_summary, indent=2, ensure_ascii=False))
CONFIG_DIR = Path(os.getenv(
    "PARSER_CONFIG_DIR",
    "/kaggle/input/models/vnphongv/government-parser-qwen3-4b-lora-artifact/other/default/1/configs"
))

print("CONFIG_DIR:", CONFIG_DIR)
print("CONFIG_DIR exists:", CONFIG_DIR.exists())

In [ ]:
def _find_adapter_dir(root: Path) -> Path | None:
    direct_candidates = [root, root / "final_adapter"]
    for candidate in direct_candidates:
        if candidate.exists() and (candidate / "adapter_config.json").exists():
            return candidate

    for config_path in root.rglob("adapter_config.json"):
        return config_path.parent

    return None


def prepare_adapter_path() -> str:
    if ADAPTER_PATH:
        adapter_path = Path(ADAPTER_PATH).expanduser().resolve()
        if adapter_path.exists() and (adapter_path / "adapter_config.json").exists():
            print(f"Using adapter folder from PARSER_ADAPTER_PATH: {adapter_path}")
            return str(adapter_path)
        raise FileNotFoundError(
            f"PARSER_ADAPTER_PATH is set but adapter_config.json was not found: {adapter_path}"
        )

    artifact_zip_path = Path(ARTIFACT_ZIP_PATH).expanduser()
    if artifact_zip_path.exists():
        extract_dir = Path(EXTRACT_DIR)
        extract_dir.mkdir(parents=True, exist_ok=True)

        print(f"Extracting adapter artifact zip: {artifact_zip_path}")
        with zipfile.ZipFile(artifact_zip_path, "r") as artifact_zip:
            artifact_zip.extractall(extract_dir)

        adapter_dir = _find_adapter_dir(extract_dir)
        if adapter_dir is not None:
            print(f"Using extracted adapter folder: {adapter_dir}")
            return str(adapter_dir)

        raise FileNotFoundError(
            f"Artifact zip was extracted to {extract_dir}, but no folder containing adapter_config.json was found. "
            "Do not use paths like artifact.zip\\final_adapter on Kaggle; extract the zip first or set PARSER_ADAPTER_PATH."
        )

    raise FileNotFoundError(
        "Could not find the parser LoRA adapter. Set PARSER_ADAPTER_PATH to an extracted final_adapter folder, "
        f"or set PARSER_ARTIFACT_ZIP to a real zip file. Current zip path: {ARTIFACT_ZIP_PATH}"
    )

In [ ]:
import json
import re
import unicodedata
from pathlib import Path
from typing import Any

def find_config_dir() -> Path:
    candidates = []

    if "CONFIG_DIR" in globals():
        candidates.append(Path(CONFIG_DIR))

    candidates.extend([
        Path("/kaggle/input/models/vnphongv/government-parser-qwen3-4b-lora-artifact/other/default/1/configs"),
        Path("/kaggle/working/government_parser_artifact/configs"),
        Path("/kaggle/working/government_parser_artifact/final_adapter/configs"),
    ])

    for path in candidates:
        if path.exists() and path.is_dir():
            return path

    root = Path("/kaggle/input")
    if root.exists():
        for path in root.rglob("configs"):
            if path.is_dir():
                return path

    raise FileNotFoundError("Could not find parser configs directory. Set PARSER_CONFIG_DIR.")

CONFIG_DIR = find_config_dir()
print("Using CONFIG_DIR:", CONFIG_DIR)

def find_config_file(stem: str) -> Path:
    candidates = [
        CONFIG_DIR / stem,
        CONFIG_DIR / f"{stem}.json",
    ]

    for path in candidates:
        if path.exists():
            return path

    matches = list(CONFIG_DIR.glob(f"{stem}*"))
    if matches:
        return matches[0]

    raise FileNotFoundError(f"Missing config file for {stem} in {CONFIG_DIR}")

def load_config_json(stem: str) -> Any:
    path = find_config_file(stem)
    with open(path, "r", encoding="utf-8") as f:
        data = json.load(f)
    print(f"Loaded {stem}: {path.name}")
    return data

raw_indicator_catalog = load_config_json("indicator_catalog.v1")
raw_country_catalog = load_config_json("country_catalog.v1")
raw_parser_enums = load_config_json("parser_enums.v1")
raw_parser_intents = load_config_json("parser_intents.v1")
raw_question_families = load_config_json("question_families.v1")

def normalize_text(text: str) -> str:
    text = str(text or "").lower().strip()
    text = unicodedata.normalize("NFD", text)
    text = "".join(ch for ch in text if unicodedata.category(ch) != "Mn")
    text = text.replace("_", " ")
    text = re.sub(r"[^a-z0-9%\s/.-]", " ", text)
    text = re.sub(r"\s+", " ", text)
    return text.strip()

def as_list(value: Any) -> list:
    if value is None:
        return []
    if isinstance(value, list):
        return value
    if isinstance(value, tuple):
        return list(value)
    if isinstance(value, str):
        return [value]
    return []

def normalize_indicator_catalog(raw: Any) -> dict[str, dict]:
    records = raw.get("indicators", []) if isinstance(raw, dict) else raw
    catalog = {}

    for row in records:
        code = row.get("code")
        if not code:
            continue

        aliases = []
        aliases.extend(as_list(row.get("aliases")))
        aliases.append(code)
        aliases.append(row.get("name") or code)

        hint = row.get("question_templates_hint") or {}
        aliases.extend([
            hint.get("vi"),
            hint.get("vi_no_diacritics"),
            hint.get("en"),
            hint.get("technical"),
        ])

        clean_aliases = []
        seen = set()
        for alias in aliases:
            alias = str(alias or "").strip()
            if alias and alias not in seen:
                seen.add(alias)
                clean_aliases.append(alias)

        catalog[code] = {
            "code": code,
            "name": row.get("name") or code,
            "category": row.get("category"),
            "unit": row.get("unit"),
            "gold_table": row.get("gold_table"),
            "analytics_table": row.get("analytics_table"),
            "supports_trend": row.get("supports_trend"),
            "supports_anomaly": row.get("supports_anomaly"),
            "used_for_cluster": row.get("used_for_cluster"),
            "description": row.get("description") or "",
            "aliases": clean_aliases,
        }

    return catalog

def normalize_country_catalog(raw: Any) -> dict[str, dict]:
    records = raw.get("countries", []) if isinstance(raw, dict) else raw
    catalog = {}

    for row in records:
        code = row.get("code")
        if not code:
            continue

        code = str(code).upper()

        aliases = []
        aliases.extend(as_list(row.get("aliases")))
        aliases.append(code)
        aliases.append(row.get("name") or code)

        hint = row.get("question_templates_hint") or {}
        aliases.extend([
            hint.get("vi"),
            hint.get("vi_no_diacritics"),
            hint.get("en"),
            hint.get("iso3"),
        ])

        clean_aliases = []
        seen = set()
        for alias in aliases:
            alias = str(alias or "").strip()
            if alias and alias not in seen:
                seen.add(alias)
                clean_aliases.append(alias)

        catalog[code] = {
            "code": code,
            "name": row.get("name") or code,
            "region": row.get("region"),
            "aliases": clean_aliases,
        }

    return catalog

def normalize_intents(raw: Any) -> list[str]:
    if isinstance(raw, list):
        return [str(item) for item in raw]

    if isinstance(raw, dict):
        for key in ["intents", "allowed_intents", "items", "data"]:
            if isinstance(raw.get(key), list):
                return [str(item.get("code") or item.get("intent") or item) if isinstance(item, dict) else str(item) for item in raw[key]]

    return []

def normalize_question_families(raw: Any) -> tuple[list[str], dict[str, list[str]], dict[str, dict]]:
    records = raw.get("families", []) if isinstance(raw, dict) else raw
    families = []
    mapping = {}
    metadata = {}

    for row in records:
        if isinstance(row, str):
            family_id = row
            intent = None
            row_meta = {"id": family_id}
        else:
            family_id = row.get("id") or row.get("code") or row.get("question_family") or row.get("family")
            intent = row.get("intent")
            row_meta = row

        if not family_id:
            continue

        family_id = str(family_id)
        families.append(family_id)
        metadata[family_id] = row_meta

        if intent:
            intent = str(intent)
            mapping.setdefault(intent, [])
            if family_id not in mapping[intent]:
                mapping[intent].append(family_id)

    deduped = []
    seen = set()
    for family in families:
        if family not in seen:
            seen.add(family)
            deduped.append(family)

    return deduped, mapping, metadata

INDICATOR_CATALOG = normalize_indicator_catalog(raw_indicator_catalog)
COUNTRY_CATALOG = normalize_country_catalog(raw_country_catalog)
ALLOWED_INTENTS = normalize_intents(raw_parser_intents)
QUESTION_FAMILIES, QUESTION_FAMILY_BY_INTENT, QUESTION_FAMILY_METADATA = normalize_question_families(raw_question_families)
PARSER_ENUMS = raw_parser_enums

print("Catalog loaded")
print("Total indicators:", len(INDICATOR_CATALOG))
print("Total countries:", len(COUNTRY_CATALOG))
print("Total intents:", len(ALLOWED_INTENTS))
print("Total question_families:", len(QUESTION_FAMILIES))
print("Intent-family mapping counts:")
for intent, families in sorted(QUESTION_FAMILY_BY_INTENT.items()):
    print(f"{intent}: {len(families)}")

def alias_in_text(normalized_text: str, alias: str) -> bool:
    normalized_alias = normalize_text(alias)
    if not normalized_alias:
        return False

    if len(normalized_alias) <= 3:
        return re.search(rf"(^|\s){re.escape(normalized_alias)}($|\s)", normalized_text) is not None

    return normalized_alias in normalized_text

def score_alias(normalized_text: str, alias: str) -> float:
    normalized_alias = normalize_text(alias)
    if not alias_in_text(normalized_text, alias):
        return 0.0

    if normalized_text == normalized_alias:
        return 1.0

    return min(0.99, 0.55 + len(normalized_alias) / 60)

def resolve_candidate_indicators(message: str, limit: int = 8) -> list[dict]:
    normalized_message = normalize_text(message)
    matches = []

    for code, meta in INDICATOR_CATALOG.items():
        best_score = 0.0
        best_alias = ""

        aliases = []
        aliases.append(code)
        aliases.append(meta.get("name") or code)
        aliases.extend(as_list(meta.get("aliases")))

        for alias in aliases:
            score = score_alias(normalized_message, alias)
            if score > best_score:
                best_score = score
                best_alias = alias

        if best_score >= 0.6:
            matches.append({
                "code": code,
                "name": meta.get("name") or code,
                "matched_alias": best_alias,
                "confidence": round(best_score, 3),
            })

    matches.sort(key=lambda item: item["confidence"], reverse=True)

    result = []
    seen = set()
    for item in matches:
        if item["code"] not in seen:
            seen.add(item["code"])
            result.append(item)

    return result[:limit]

def resolve_candidate_countries(message: str, limit: int = 12) -> list[dict]:
    normalized_message = normalize_text(message)
    raw_message = str(message or "")
    matches = []

    raw_tokens = set(re.findall(r"\b[A-Z]{3}\b", raw_message))

    for code, meta in COUNTRY_CATALOG.items():
        best_score = 0.0
        best_alias = ""

        aliases = []
        aliases.append(meta.get("name") or code)
        aliases.extend(as_list(meta.get("aliases")))

        if code in raw_tokens:
            aliases.append(code)

        for alias in aliases:
            alias_text = str(alias or "").strip()

            if alias_text.upper() == code and code not in raw_tokens:
                continue

            score = score_alias(normalized_message, alias_text)
            if score > best_score:
                best_score = score
                best_alias = alias_text

        if best_score >= 0.6:
            matches.append({
                "code": code,
                "name": meta.get("name") or code,
                "matched_alias": best_alias,
                "confidence": round(best_score, 3),
            })

    matches.sort(key=lambda item: item["confidence"], reverse=True)

    result = []
    seen = set()
    for item in matches:
        if item["code"] not in seen:
            seen.add(item["code"])
            result.append(item)

    return result[:limit]

def resolve_candidate_years(message: str) -> list[int]:
    years = re.findall(r"\b(?:19|20)\d{2}\b", message)
    return sorted({int(year) for year in years})

def allowed_intent(name: str) -> str | None:
    if name in ALLOWED_INTENTS:
        return name

    target = name.lower()
    for intent in ALLOWED_INTENTS:
        if intent.lower() == target:
            return intent

    return None

def add_intent(result: list[str], name: str) -> None:
    intent = allowed_intent(name)
    if intent and intent not in result:
        result.append(intent)

def families_for_intents(intents: list[str]) -> list[str]:
    result = []
    seen = set()

    for intent in intents:
        for family in QUESTION_FAMILY_BY_INTENT.get(intent, []):
            if family not in seen:
                seen.add(family)
                result.append(family)

    return result

def prioritize_families(families: list[str], preferred: list[str]) -> list[str]:
    result = []
    seen = set()

    for family in preferred:
        if family in families and family not in seen:
            seen.add(family)
            result.append(family)

    for family in families:
        if family not in seen:
            seen.add(family)
            result.append(family)

    return result

def infer_candidate_intents_and_families(
    message: str,
    candidate_indicators: list[dict],
    candidate_countries: list[dict],
    candidate_years: list[int],
) -> tuple[list[str], list[str]]:
    normalized = normalize_text(message)
    intents = []

    has_compare = any(token in normalized for token in ["so sanh", "compare", "doi chieu"])
    has_ranking = any(token in normalized for token in ["top", "cao nhat", "thap nhat", "xep hang", "ranking"])
    has_trend = any(token in normalized for token in ["xu huong", "trend", "dien bien", "chuoi thoi gian"])
    has_anomaly = any(token in normalized for token in ["bat thuong", "anomaly", "canh bao"])
    has_coverage = any(token in normalized for token in ["coverage", "co du lieu", "thieu du lieu", "bao phu"])
    has_forecast = any(token in normalized for token in ["du bao", "forecast", "arima", "predict", "prediction"])

    if has_forecast:
        add_intent(intents, "UNSUPPORTED")

    if has_compare and len(candidate_countries) >= 2:
        add_intent(intents, "COMPARE_COUNTRIES")

    if has_ranking:
        add_intent(intents, "RANKING")

    if has_trend:
        add_intent(intents, "TREND_ANALYSIS")
        add_intent(intents, "TIME_SERIES")

    if has_anomaly:
        add_intent(intents, "ANOMALY_DETECTION")

    if has_coverage:
        add_intent(intents, "COVERAGE")

    if (has_compare or has_ranking or has_trend or has_anomaly) and not candidate_indicators:
        add_intent(intents, "NEED_CLARIFICATION")

    if not intents and candidate_indicators:
        if candidate_countries and candidate_years:
            add_intent(intents, "VALUE_LOOKUP")
        elif candidate_countries:
            add_intent(intents, "TIME_SERIES")

    if not intents:
        add_intent(intents, "NEED_CLARIFICATION")

    families = families_for_intents(intents)

    preferred = []
    if "COMPARE_COUNTRIES" in intents:
        if len(candidate_years) >= 2:
            preferred.append("compare_countries_period")
        elif len(candidate_years) == 1:
            preferred.append("compare_countries_single_year")

    if "RANKING" in intents:
        if any(token in normalized for token in ["thap nhat", "bottom", "lowest"]):
            preferred.append("ranking_bottom_n")
        else:
            preferred.append("ranking_top_n")

    families = prioritize_families(families, preferred)

    return intents, families

def build_grounding_candidates(message: str, context: dict | None = None) -> dict:
    candidate_indicators = resolve_candidate_indicators(message)
    candidate_countries = resolve_candidate_countries(message)
    detected_years = resolve_candidate_years(message)
    candidate_intents, candidate_question_families = infer_candidate_intents_and_families(
        message=message,
        candidate_indicators=candidate_indicators,
        candidate_countries=candidate_countries,
        candidate_years=detected_years,
    )

    return {
        "candidate_indicators": candidate_indicators,
        "candidate_countries": candidate_countries,
        "detected_years": detected_years,
        "candidate_intents": candidate_intents,
        "candidate_question_families": candidate_question_families,
    }

In [ ]:
REQUIRED_FIELDS = [
    "intent",
    "question_family",
    "indicators",
    "countries",
    "country_groups",
    "start_year",
    "end_year",
    "relative_time",
    "event_time",
    "ranking_order",
    "limit",
    "threshold",
    "aggregation",
    "chart_preference",
    "needs_clarification",
    "clarification_questions",
    "confidence",
]

DEFAULT_FALLBACK_PARSED = {
    "intent": "NEED_CLARIFICATION",
    "question_family": "parser_error",
    "indicators": [],
    "countries": [],
    "country_groups": [],
    "start_year": None,
    "end_year": None,
    "relative_time": None,
    "event_time": None,
    "ranking_order": None,
    "limit": None,
    "threshold": None,
    "aggregation": None,
    "chart_preference": "none",
    "needs_clarification": True,
    "clarification_questions": [
        "Mình chưa phân tích chắc chắn được câu hỏi. Bạn có thể nói rõ chỉ số, quốc gia và giai đoạn cần phân tích không?"
    ],
    "confidence": 0.0,
}

In [ ]:
def build_fallback_parsed(reason: str) -> dict[str, Any]:
    parsed = dict(DEFAULT_FALLBACK_PARSED)
    parsed["clarification_questions"] = list(DEFAULT_FALLBACK_PARSED.get("clarification_questions", []))

    if reason == "invalid_json":
        parsed["question_family"] = "parser_invalid_json"
        parsed["clarification_questions"] = [
            "Mình chưa đọc được kết quả parser dưới dạng JSON hợp lệ. Bạn có thể nói rõ chỉ số, quốc gia và giai đoạn cần phân tích không?"
        ]
    elif reason == "schema_error":
        parsed["question_family"] = "parser_schema_error"
        parsed["clarification_questions"] = [
            "Kết quả parser chưa đúng schema. Bạn có thể nói rõ chỉ số, quốc gia và giai đoạn cần phân tích không?"
        ]

    return parsed

In [ ]:
def extract_first_json_object(text: str) -> str | None:
    if not text:
        return None

    cleaned = text.strip()
    if cleaned.startswith("```"):
        lines = cleaned.splitlines()
        if lines and lines[0].strip().startswith("```"):
            lines = lines[1:]
        if lines and lines[-1].strip() == "```":
            lines = lines[:-1]
        cleaned = "\n".join(lines).strip()

    start = cleaned.find("{")
    if start == -1:
        return None

    depth = 0
    in_string = False
    escaped = False

    for index in range(start, len(cleaned)):
        char = cleaned[index]

        if in_string:
            if escaped:
                escaped = False
            elif char == "\\":
                escaped = True
            elif char == '"':
                in_string = False
            continue

        if char == '"':
            in_string = True
        elif char == "{":
            depth += 1
        elif char == "}":
            depth -= 1
            if depth == 0:
                return cleaned[start : index + 1]

    return None


def parse_model_json(raw_text: str) -> tuple[dict | None, bool, str | None]:
    json_text = extract_first_json_object(raw_text)
    if json_text is None:
        return None, False, "no_json_object_found"

    try:
        parsed = json.loads(json_text)
    except json.JSONDecodeError as exc:
        return None, False, str(exc)

    if not isinstance(parsed, dict):
        return None, False, "json_value_is_not_object"

    return parsed, True, None


def basic_schema_check(parsed: dict) -> tuple[bool, list[str]]:
    errors: list[str] = []

    missing_fields = [field for field in REQUIRED_FIELDS if field not in parsed]
    if missing_fields:
        errors.append(f"missing_required_fields: {missing_fields}")

    if "intent" in parsed and not isinstance(parsed["intent"], str):
        errors.append("intent_must_be_str")
    if "question_family" in parsed and not isinstance(parsed["question_family"], str):
        errors.append("question_family_must_be_str")

    for field in ["indicators", "countries", "country_groups", "clarification_questions"]:
        if field in parsed and not isinstance(parsed[field], list):
            errors.append(f"{field}_must_be_list")

    if "needs_clarification" in parsed and not isinstance(parsed["needs_clarification"], bool):
        errors.append("needs_clarification_must_be_bool")

    if "confidence" in parsed and parsed["confidence"] is not None and not isinstance(parsed["confidence"], (int, float)):
        errors.append("confidence_must_be_number_or_null")

    return len(errors) == 0, errors

In [ ]:
def build_parser_messages(
    message: str,
    context: dict | None = None,
    candidates: dict | None = None,
) -> list[dict]:
    candidates = candidates or {}

    system_prompt = """
You are a semantic parser for a Government Economic Indicator AI Agent.

Return only one valid JSON object matching ParsedQuery v1.
Do not answer the user question.
Do not write SQL.
Do not query data.
Do not invent missing indicators, countries, years, intents, or question families.

Only use valid indicators, countries, intents, and question families from the provided candidate/catalog section.
If a required slot is missing from the user message, return NEED_CLARIFICATION.
Prefer NEED_CLARIFICATION over guessing.

Required JSON fields:
intent, question_family, indicators, countries, country_groups, start_year, end_year,
relative_time, event_time, ranking_order, limit, threshold, aggregation,
chart_preference, needs_clarification, clarification_questions, confidence.

Canonical family preference:
- COMPARE_COUNTRIES with >=2 countries and a year range -> compare_countries_period if available.
- COMPARE_COUNTRIES with >=2 countries and a single year -> compare_countries_single_year if available.
- RANKING top/highest -> ranking_top_n if available.
- RANKING bottom/lowest -> ranking_bottom_n if available.

Use null for unknown scalar fields.
Use [] for unknown list fields.
Use "none" for chart_preference if no chart is requested.
Output JSON only.
""".strip()

    payload = {
        "user_message": message,
        "context": context,
        "allowed_intents": ALLOWED_INTENTS,
        "candidate_indicators": candidates.get("candidate_indicators", []),
        "candidate_countries": candidates.get("candidate_countries", []),
        "detected_years": candidates.get("detected_years", []),
        "candidate_intents": candidates.get("candidate_intents", []),
        "candidate_question_families": candidates.get("candidate_question_families", []),
        "instructions": [
            "Return JSON only.",
            "Choose intent only from candidate_intents when candidate_intents is not empty.",
            "Choose question_family only from candidate_question_families when candidate_question_families is not empty.",
            "Do not choose an indicator outside candidate_indicators unless the user explicitly wrote a valid indicator code.",
            "Do not choose a country outside candidate_countries unless the user explicitly wrote a valid country code.",
            "Do not fabricate years.",
            "If comparison/ranking/value/trend query lacks indicator, return NEED_CLARIFICATION.",
            "If ranking/value lookup lacks year/time, return NEED_CLARIFICATION.",
        ],
    }

    user_prompt = json.dumps(payload, ensure_ascii=False, indent=2)

    return [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt},
    ]

In [ ]:
tokenizer = None
model = None
adapter_real_path = None
load_error = None
model_loaded = False
tokenizer_loaded = False
adapter_loaded = False


def load_parser_model():
    global tokenizer, model, adapter_real_path, load_error
    global model_loaded, tokenizer_loaded, adapter_loaded

    load_error = None
    model_loaded = False
    tokenizer_loaded = False
    adapter_loaded = False

    try:
        adapter_real_path = prepare_adapter_path()

        try:
            tokenizer = AutoTokenizer.from_pretrained(adapter_real_path, trust_remote_code=True)
            print(f"Loaded tokenizer from adapter path: {adapter_real_path}")
        except Exception as tokenizer_error:
            print(f"Could not load tokenizer from adapter path: {tokenizer_error}")
            print(f"Falling back to base tokenizer: {BASE_MODEL_ID}")
            tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_ID, trust_remote_code=True)

        if tokenizer.pad_token is None:
            tokenizer.pad_token = tokenizer.eos_token
        tokenizer_loaded = True

        quantization_config = None
        if LOAD_IN_4BIT and torch.cuda.is_available():
            quantization_config = BitsAndBytesConfig(
                load_in_4bit=True,
                bnb_4bit_quant_type="nf4",
                bnb_4bit_compute_dtype=torch.bfloat16 if torch.cuda.is_available() else torch.float32,
                bnb_4bit_use_double_quant=True,
            )
        elif LOAD_IN_4BIT:
            print("4-bit loading was requested, but CUDA is not available. Loading without 4-bit quantization.")

        base_model = AutoModelForCausalLM.from_pretrained(
            BASE_MODEL_ID,
            device_map="auto",
            quantization_config=quantization_config,
            trust_remote_code=True,
        )

        model = PeftModel.from_pretrained(base_model, adapter_real_path)
        model.eval()

        adapter_loaded = True
        model_loaded = True

        print("Parser model loaded successfully.")
        return model
    except Exception as exc:
        load_error = f"{type(exc).__name__}: {exc}"
        print(f"Parser model load failed: {load_error}")
        raise


try:
    load_parser_model()
except Exception:
    print("Continuing with degraded service state. /health will expose load_error and /parse will return HTTP 503 until the model loads successfully.")

In [ ]:
def generate_parser_output(
    message: str,
    context: dict | None = None,
    candidates: dict | None = None,
) -> str:
    if model is None or tokenizer is None:
        raise RuntimeError("Parser model is not loaded")

    messages = build_parser_messages(message, context, candidates)

    if hasattr(tokenizer, "apply_chat_template"):
        try:
            encoded = tokenizer.apply_chat_template(
                messages,
                tokenize=True,
                add_generation_prompt=True,
                return_tensors="pt",
                return_dict=True,
            )
        except TypeError:
            input_ids = tokenizer.apply_chat_template(
                messages,
                tokenize=True,
                add_generation_prompt=True,
                return_tensors="pt",
            )
            encoded = {"input_ids": input_ids}
    else:
        manual_prompt = "\n".join([f"{m['role'].upper()}: {m['content']}" for m in messages])
        encoded = tokenizer(manual_prompt, return_tensors="pt")

    if isinstance(encoded, torch.Tensor):
        encoded = {"input_ids": encoded}
    else:
        encoded = dict(encoded)

    model_inputs = {}
    for key in ["input_ids", "attention_mask"]:
        value = encoded.get(key)
        if value is not None:
            model_inputs[key] = value.to(model.device)

    input_ids = model_inputs["input_ids"]

    generation_kwargs = {
        **model_inputs,
        "max_new_tokens": MAX_NEW_TOKENS,
        "do_sample": False,
        "pad_token_id": tokenizer.pad_token_id or tokenizer.eos_token_id,
        "eos_token_id": tokenizer.eos_token_id,
    }

    with torch.no_grad():
        output_ids = model.generate(**generation_kwargs)

    generated_ids = output_ids[0][input_ids.shape[-1]:]
    raw_text = tokenizer.decode(generated_ids, skip_special_tokens=True)

    return raw_text.strip()

In [ ]:
class ParseRequest(BaseModel):
    message: str
    context: dict[str, Any] | None = None
    debug: bool = False


class ParseResponse(BaseModel):
    parsed: dict[str, Any]
    raw_model_output: str | None = None
    valid_json: bool
    schema_pass: bool
    catalog_pass: bool
    safe_to_execute: bool
    fallback_reason: str | None = None
    repairs_applied: list[str] = Field(default_factory=list)
    candidates: dict[str, Any] = Field(default_factory=dict)
    latency_ms: int


app = FastAPI(title="Government Parser Model Service", version=SERVICE_VERSION)


@app.get("/health")
def health() -> dict[str, Any]:
    if model_loaded and tokenizer_loaded and adapter_loaded:
        status = "ok"
    elif load_error:
        status = "error"
    else:
        status = "degraded"

    return {
        "status": status,
        "service": "parser-model-service",
        "version": SERVICE_VERSION,
        "model_loaded": model_loaded,
        "tokenizer_loaded": tokenizer_loaded,
        "adapter_loaded": adapter_loaded,
        "base_model_id": BASE_MODEL_ID,
        "adapter_path": adapter_real_path,
        "load_error": load_error,
    }


@app.post("/parse", response_model=ParseResponse)
def parse(req: ParseRequest):
    start = time.time()

    message = (req.message or "").strip()
    if not message:
        raise HTTPException(status_code=400, detail="message is required")

    if not model_loaded or model is None or tokenizer is None:
        raise HTTPException(
            status_code=503,
            detail={
                "message": "Parser model is not loaded",
                "load_error": load_error,
            },
        )

    candidates = build_grounding_candidates(message, req.context)

    raw_output = generate_parser_output(
        message=message,
        context=req.context,
        candidates=candidates,
    )

    parsed, valid_json, json_error = parse_model_json(raw_output)

    if not valid_json or parsed is None:
        parsed = build_fallback_parsed("invalid_json")
        schema_pass = False
        fallback_reason = "invalid_json"
    else:
        schema_pass, schema_errors = basic_schema_check(parsed)
        if not schema_pass:
            fallback_reason = "schema_error"
        else:
            fallback_reason = "catalog_validator_not_implemented_phase3"

    latency_ms = int((time.time() - start) * 1000)

    return ParseResponse(
        parsed=parsed,
        raw_model_output=raw_output if req.debug else None,
        valid_json=valid_json,
        schema_pass=schema_pass,
        catalog_pass=False,
        safe_to_execute=False,
        fallback_reason=fallback_reason,
        repairs_applied=[],
        candidates=candidates,
        latency_ms=latency_ms,
    )

In [ ]:
try:
    raw_parsed_query_schema
except NameError:
    raw_parsed_query_schema = load_config_json("parsed_query_schema.v1")

try:
    import jsonschema
except Exception:
    jsonschema = None


EXECUTABLE_INTENTS = {
    "COMPARE_COUNTRIES",
    "COMPARE_INDICATORS",
    "RANKING",
    "RANK_BY_CHANGE",
    "TIME_SERIES",
    "VALUE_LOOKUP",
    "TREND_ANALYSIS",
    "ANOMALY_DETECTION",
    "COVERAGE",
}

REQUIRES_INDICATOR_INTENTS = {
    "COMPARE_COUNTRIES",
    "COMPARE_INDICATORS",
    "RANKING",
    "RANK_BY_CHANGE",
    "TIME_SERIES",
    "VALUE_LOOKUP",
    "TREND_ANALYSIS",
    "ANOMALY_DETECTION",
}

REQUIRES_TIME_INTENTS = {
    "VALUE_LOOKUP",
    "RANKING",
    "COMPARE_COUNTRIES",
    "TIME_SERIES",
    "TREND_ANALYSIS",
    "RANK_BY_CHANGE",
    "ANOMALY_DETECTION",
}


def safe_list(value: Any) -> list:
    if isinstance(value, list):
        return value
    if value is None:
        return []
    return [value]


def candidate_codes(candidates: dict | None, key: str) -> set[str]:
    values = set()
    items = (candidates or {}).get(key) or []

    for item in items:
        if isinstance(item, dict):
            code = item.get("code")
            if code:
                values.add(str(code))
        elif item is not None:
            values.add(str(item))

    return values


def detected_year_set(candidates: dict | None) -> set[int]:
    result = set()
    for item in (candidates or {}).get("detected_years") or []:
        try:
            result.add(int(item))
        except Exception:
            pass
    return result


def get_candidate_intents(candidates: dict | None) -> set[str]:
    return {str(item) for item in ((candidates or {}).get("candidate_intents") or [])}


def get_candidate_families(candidates: dict | None) -> set[str]:
    return {str(item) for item in ((candidates or {}).get("candidate_question_families") or [])}


def normalize_parsed_basic(parsed: dict) -> dict:
    normalized = dict(DEFAULT_FALLBACK_PARSED)
    normalized.update(parsed or {})

    normalized["intent"] = str(normalized.get("intent") or "").strip()
    normalized["question_family"] = str(normalized.get("question_family") or "").strip()

    normalized["indicators"] = [
        str(item).strip()
        for item in safe_list(normalized.get("indicators"))
        if item is not None and str(item).strip()
    ]

    normalized["countries"] = [
        str(item).strip().upper()
        for item in safe_list(normalized.get("countries"))
        if item is not None and str(item).strip()
    ]

    normalized["country_groups"] = [
        str(item).strip()
        for item in safe_list(normalized.get("country_groups"))
        if item is not None and str(item).strip()
    ]

    normalized["clarification_questions"] = [
        str(item).strip()
        for item in safe_list(normalized.get("clarification_questions"))
        if item is not None and str(item).strip()
    ]

    for field in ["start_year", "end_year", "limit"]:
        value = normalized.get(field)
        if value is not None:
            try:
                normalized[field] = int(value)
            except Exception:
                pass

    value = normalized.get("confidence")
    try:
        normalized["confidence"] = float(value)
    except Exception:
        normalized["confidence"] = 0.0

    return normalized


def schema_validate_full(parsed: dict) -> tuple[bool, list[str]]:
    schema_pass, schema_errors = basic_schema_check(parsed)

    if jsonschema is None:
        return schema_pass, schema_errors

    try:
        jsonschema.validate(instance=parsed, schema=raw_parsed_query_schema)
    except Exception as exc:
        schema_errors.append(f"jsonschema_error:{str(exc)}")

    return len(schema_errors) == 0, schema_errors


def family_allowed_for_intent(question_family: str, intent: str) -> bool:
    if not question_family or not intent:
        return False

    if question_family not in QUESTION_FAMILIES:
        return False

    mapped = QUESTION_FAMILY_BY_INTENT.get(intent, [])
    if mapped:
        return question_family in mapped

    if intent == "NEED_CLARIFICATION":
        return question_family.startswith("missing_") or question_family.startswith("ambiguous_")

    if intent == "UNSUPPORTED":
        return question_family.startswith("unsupported_")

    return True


def derive_safe_question_family(parsed: dict, candidates: dict | None) -> str | None:
    intent = parsed.get("intent")
    candidate_families = get_candidate_families(candidates)

    def allowed(family: str) -> str | None:
        if family in QUESTION_FAMILIES and family_allowed_for_intent(family, intent):
            if not candidate_families or family in candidate_families:
                return family
        return None

    if intent == "COMPARE_COUNTRIES":
        countries = parsed.get("countries") or []
        start_year = parsed.get("start_year")
        end_year = parsed.get("end_year")

        if len(countries) >= 2:
            if start_year is not None and end_year is not None and start_year == end_year:
                return allowed("compare_countries_single_year")
            if start_year is not None and end_year is not None:
                return allowed("compare_countries_period")

    if intent == "RANKING":
        order = parsed.get("ranking_order")
        if order == "desc":
            return allowed("ranking_top_n")
        if order == "asc":
            return allowed("ranking_bottom_n")

    if intent == "TREND_ANALYSIS":
        countries = parsed.get("countries") or []
        start_year = parsed.get("start_year")
        end_year = parsed.get("end_year")
        if len(countries) == 1 and start_year is not None and end_year is not None:
            return allowed("trend_single_country_period")
        if len(countries) > 1 and start_year is not None and end_year is not None:
            return allowed("trend_multi_country_period")

    if intent == "TIME_SERIES":
        countries = parsed.get("countries") or []
        start_year = parsed.get("start_year")
        end_year = parsed.get("end_year")
        if len(countries) == 1 and start_year is not None and end_year is not None:
            return allowed("time_series_single_country_period")
        if len(countries) > 1 and start_year is not None and end_year is not None:
            return allowed("time_series_multi_country_period")

    return None


def make_clarification(parsed: dict, question: str, family: str, reason: str) -> dict:
    updated = dict(parsed)
    updated["intent"] = "NEED_CLARIFICATION"
    updated["question_family"] = family
    updated["needs_clarification"] = True
    updated["clarification_questions"] = [question]
    updated["confidence"] = min(float(updated.get("confidence") or 0.6), 0.6)
    return updated


def validate_catalog_and_guard(parsed: dict, message: str, candidates: dict | None = None) -> dict:
    errors = []
    repairs_applied = []

    normalized = normalize_parsed_basic(parsed)

    intent = normalized.get("intent")
    question_family = normalized.get("question_family")

    candidate_indicator_codes = candidate_codes(candidates, "candidate_indicators")
    candidate_country_codes = candidate_codes(candidates, "candidate_countries")
    candidate_intents = get_candidate_intents(candidates)
    candidate_families = get_candidate_families(candidates)
    years_in_user = detected_year_set(candidates)

    if intent not in ALLOWED_INTENTS:
        errors.append(f"invalid_intent:{intent}")

    if candidate_intents and intent not in candidate_intents:
        if intent not in {"NEED_CLARIFICATION", "UNSUPPORTED"}:
            errors.append(f"intent_not_supported_by_candidates:{intent}")

    if question_family not in QUESTION_FAMILIES:
        derived_family = derive_safe_question_family(normalized, candidates)
        if derived_family:
            repairs_applied.append(f"question_family:{question_family}->{derived_family}")
            normalized["question_family"] = derived_family
            question_family = derived_family
        else:
            errors.append(f"invalid_question_family:{question_family}")

    if question_family in QUESTION_FAMILIES:
        if not family_allowed_for_intent(question_family, intent):
            derived_family = derive_safe_question_family(normalized, candidates)
            if derived_family:
                repairs_applied.append(f"question_family:{question_family}->{derived_family}")
                normalized["question_family"] = derived_family
                question_family = derived_family
            else:
                errors.append(f"question_family_not_allowed_for_intent:{question_family}:{intent}")

    if candidate_families and question_family not in candidate_families:
        if intent not in {"NEED_CLARIFICATION", "UNSUPPORTED"}:
            derived_family = derive_safe_question_family(normalized, candidates)
            if derived_family:
                repairs_applied.append(f"question_family:{question_family}->{derived_family}")
                normalized["question_family"] = derived_family
                question_family = derived_family
            else:
                errors.append(f"question_family_not_supported_by_candidates:{question_family}")

    for indicator in normalized.get("indicators", []):
        if indicator not in INDICATOR_CATALOG:
            errors.append(f"invalid_indicator:{indicator}")
        elif candidate_indicator_codes and indicator not in candidate_indicator_codes:
            errors.append(f"indicator_not_mentioned_by_user:{indicator}")
        elif not candidate_indicator_codes:
            errors.append(f"indicator_without_user_evidence:{indicator}")

    for country in normalized.get("countries", []):
        if country not in COUNTRY_CATALOG:
            errors.append(f"invalid_country:{country}")
        elif candidate_country_codes and country not in candidate_country_codes:
            errors.append(f"country_not_mentioned_by_user:{country}")

    if intent == "UNSUPPORTED":
        normalized["needs_clarification"] = False
        normalized["clarification_questions"] = []
        return {
            "parsed": normalized,
            "catalog_pass": True,
            "safe_to_execute": False,
            "fallback_reason": "unsupported_request",
            "repairs_applied": repairs_applied,
            "validation_errors": errors,
        }

    if intent == "NEED_CLARIFICATION" or normalized.get("needs_clarification") is True:
        normalized["intent"] = "NEED_CLARIFICATION"
        normalized["needs_clarification"] = True
        if not normalized.get("clarification_questions"):
            normalized["clarification_questions"] = [
                "Bạn muốn phân tích chỉ số nào, quốc gia nào và trong giai đoạn nào?"
            ]
        return {
            "parsed": normalized,
            "catalog_pass": len(errors) == 0,
            "safe_to_execute": False,
            "fallback_reason": "needs_clarification",
            "repairs_applied": repairs_applied,
            "validation_errors": errors,
        }

    if intent in REQUIRES_INDICATOR_INTENTS:
        if not normalized.get("indicators") or not candidate_indicator_codes:
            fallback = make_clarification(
                normalized,
                "Bạn muốn so sánh/phân tích chỉ số nào?",
                "missing_indicator",
                "missing_indicator_in_user_message",
            )
            return {
                "parsed": fallback,
                "catalog_pass": len(errors) == 0,
                "safe_to_execute": False,
                "fallback_reason": "missing_indicator_in_user_message",
                "repairs_applied": repairs_applied,
                "validation_errors": errors,
            }

    if intent in REQUIRES_TIME_INTENTS:
        start_year = normalized.get("start_year")
        end_year = normalized.get("end_year")

        if start_year is None or end_year is None or not years_in_user:
            fallback = make_clarification(
                normalized,
                "Bạn muốn phân tích trong năm nào hoặc giai đoạn nào?",
                "ambiguous_time_range",
                "missing_time_in_user_message",
            )
            return {
                "parsed": fallback,
                "catalog_pass": len(errors) == 0,
                "safe_to_execute": False,
                "fallback_reason": "missing_time_in_user_message",
                "repairs_applied": repairs_applied,
                "validation_errors": errors,
            }

        if isinstance(start_year, int) and start_year not in years_in_user:
            errors.append(f"start_year_not_mentioned_by_user:{start_year}")

        if isinstance(end_year, int) and end_year not in years_in_user:
            errors.append(f"end_year_not_mentioned_by_user:{end_year}")

    if errors:
        fallback = make_clarification(
            normalized,
            "Mình chưa xác thực được chỉ số, quốc gia hoặc thời gian trong câu hỏi. Bạn có thể nói rõ hơn không?",
            "ambiguous_indicator",
            "catalog_validation_failed",
        )
        return {
            "parsed": fallback,
            "catalog_pass": False,
            "safe_to_execute": False,
            "fallback_reason": "catalog_validation_failed",
            "repairs_applied": repairs_applied,
            "validation_errors": errors,
        }

    if intent not in EXECUTABLE_INTENTS:
        return {
            "parsed": normalized,
            "catalog_pass": True,
            "safe_to_execute": False,
            "fallback_reason": "non_executable_intent",
            "repairs_applied": repairs_applied,
            "validation_errors": [],
        }

    return {
        "parsed": normalized,
        "catalog_pass": True,
        "safe_to_execute": True,
        "fallback_reason": None,
        "repairs_applied": repairs_applied,
        "validation_errors": [],
    }

In [ ]:
class ParseRequest(BaseModel):
    message: str
    context: dict[str, Any] | None = None
    debug: bool = False


class ParseResponse(BaseModel):
    parsed: dict[str, Any]
    raw_model_output: str | None = None
    valid_json: bool
    schema_pass: bool
    catalog_pass: bool
    safe_to_execute: bool
    fallback_reason: str | None = None
    repairs_applied: list[str] = Field(default_factory=list)
    candidates: dict[str, Any] = Field(default_factory=dict)
    latency_ms: int

In [ ]:
test_cases = [
    "So sánh nợ công Việt Nam và Thái Lan từ 2010 đến 2023",
    "Top 10 nước có lạm phát CPI cao nhất năm 2020",
    "Xu hướng thất nghiệp của Việt Nam từ 2000 đến 2022",
    "So sánh Việt Nam với Thái Lan",
    "Dự báo GDP Việt Nam 2030 bằng mô hình ARIMA",
]

for message in test_cases:
    print("=" * 100)
    print("MESSAGE:", message)

    candidates = build_grounding_candidates(message)
    print("CANDIDATES:")
    print(json.dumps(candidates, ensure_ascii=False, indent=2))

    if not model_loaded:
        print("SKIP model inference because model is not loaded.")
        print("load_error:", load_error)
        continue

    raw = generate_parser_output(message, None, candidates)
    parsed, valid_json, json_error = parse_model_json(raw)

    print("RAW:")
    print(raw)

    print("VALID_JSON:", valid_json)
    if json_error:
        print("JSON_ERROR:", json_error)

    print("PARSED:")
    print(json.dumps(parsed, ensure_ascii=False, indent=2))

In [ ]:
def run_api():
    uvicorn.run(app, host="0.0.0.0", port=PORT)


server_thread = threading.Thread(target=run_api, daemon=True)
server_thread.start()

print(f"FastAPI server running on port {PORT}")

In [ ]:
if NGROK_AUTHTOKEN:
    ngrok.set_auth_token(NGROK_AUTHTOKEN)
else:
    print("WARNING: NGROK_AUTHTOKEN is not set. Set os.environ['NGROK_AUTHTOKEN'] or configure it as a Kaggle Secret before starting the tunnel.")

public_url = ngrok.connect(PORT, "http")
print("Public URL:", public_url)
print(f"GET  {public_url}/health")
print(f"POST {public_url}/parse")